In [1]:
import torch
from torchvision import transforms 
from torchvision import datasets 
from transformers import ViTForImageClassification
from torch.utils.data import DataLoader

d:\xAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else 'cpu'

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [4]:
trainset = datasets.CIFAR10(
    root = "./data",
    train = True, 
    download = False,
    transform = transform
)

In [5]:
loader = DataLoader(trainset, 
                    batch_size = 32, 
                    shuffle=True)

In [6]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels = 10, 
    ignore_mismatched_sizes = True
)
model = model.to(device)

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 7801.90it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([10])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), 
                              lr = 2e-5)
model.train()

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [8]:
for epoch in range(2):
    total_loss = 0 
    for images, labels in loader: 
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(pixel_values = images, 
                        labels = labels)
        loss = outputs.loss 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: {total_loss:.2f}")

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "vit_cifer10.pth")

cuda
